## Imports 

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast
import re

## Part 1 - Data Loading and Exploratory Overview

In [7]:
df = pd.read_csv(
    r"C:\Users\user\OneDrive - Ariel University\תקייה מחשב ישן\שנה ג\סמסטר ב\למידת מכונה\פרוייקט\dataset.csv",
)
low_memory = False

C:\Users\user\AppData\Local\Temp\ipykernel_7848\3517229104.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


## Final prepare_data

In [59]:
import pandas as pd
import numpy as np
import ast
import re

def prepare_data(df):
    # ביצוע קופי ראשוני כדי להגן על הדאטה המקורית מפני דריסה
    df_clean = df.copy()
    
    # ==========================================================================
    # 1. פונקציות עזר פנימיות (Inner Helper Functions)
    # ==========================================================================
    
    # מנוע רגקס לניקוי תווים מוזרים (כמו $, *, #) וספירת מילים בעלילה
    def clean_and_count_words(text):
        if pd.isna(text):
            return 0
        text = str(text).lower()
        # מסיר כל תו שאינו אות, מספר או רווח
        clean_text = re.sub(r'[^a-z0-9\s]', '', text)
        return len(clean_text.split())

    # מנוע דינמי לפירוק וניקוי רשימות (למדינות, שפות וז'אנרים)
    def parse_categories(text):
        if pd.isna(text):
            return []
        text = str(text).lower().strip()
        if text in ["", "[]", "\\n", "unknown", "not found", "none"]:
            return []
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                items = parsed
            else:
                items = [text]
        except:
            items = re.split(r"\||/|,| and ", text)
            
        cleaned = []
        for item in items:
            item = str(item).strip()
            item = re.sub(r"\[.*?\]", "", item)
            item = re.sub(r"\(.*?\)", "", item)
            item = item.strip()
            if item and item not in ["unknown", "not found", "none"]:
                cleaned.append(item)
        return cleaned

    # ==========================================================================
    # 2. שלבי העיבוד והטיוב (Processing Steps)
    # ==========================================================================
    
    # שלב א': טיוב משתנה המטרה (דירוג) ומחיקת עמודות זליגה/לא נחוצות
    if 'averageRating' in df_clean.columns:
        df_clean['averageRating'] = pd.to_numeric(df_clean['averageRating'], errors='coerce')
        # וידוא חוקים נוקשים לציון הרגרסיה: רק ערכים חוקיים בין 1.0 ל-10.0
        df_clean = df_clean[(df_clean['averageRating'] >= 1.0) & (df_clean['averageRating'] <= 10.0)]
    
    # הסרת עמודות זליגה (מספר מצביעים והכנסות)
    leakage_cols = ['numVotes', 'BoxOffice']
    for col in leakage_cols:
        if col in df_clean.columns:
            df_clean = df_clean.drop(columns=col)

    # וידוא תקינות שמות העמודות (פותר את בעיית האותיות הגדולות/קטנות שגרמו למדינות ולשפות להיעלם)
    for col in df_clean.columns:
        col_lower = col.lower()
        if col_lower == 'genres' and col != 'genres':
            df_clean = df_clean.rename(columns={col: 'genres'})
        elif col_lower == 'country' and col != 'Country':
            df_clean = df_clean.rename(columns={col: 'Country'})
        elif col_lower == 'language' and col != 'Language':
            df_clean = df_clean.rename(columns={col: 'Language'})

    # השלמת ערכים חסרים בסיסית לעמודות המקוריות לפני עיבוד
    fill_values = {
        'Country': 'Unknown',
        'Language': 'Unknown',
        'plot': '',
        'genres': 'Unknown',
        'primaryTitle': 'Unknown'
    }
    for col, value in fill_values.items():
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna(value)

    if 'lead_actors_ids' in df_clean.columns:
        df_clean['lead_actors_ids'] = df_clean['lead_actors_ids'].apply(
            lambda x: [] if pd.isna(x) else x
        )

    # --------------------------------------------------------------------------
    # שלב ב': Feature Engineering (בניית הפיצ'רים)
    # --------------------------------------------------------------------------

    # Feature 1: Plot word count
    if 'plot' in df_clean.columns:
        df_clean['plot_word_count'] = df_clean['plot'].apply(clean_and_count_words)

    # Feature 2: Release decade
    if 'startYear' in df_clean.columns:
        df_clean['startYear'] = pd.to_numeric(df_clean['startYear'], errors='coerce')
        # הגבלה לשנים הגיוניות של קולנוע: כל מה שמחוץ לטווח הופך ל-NaN
        df_clean.loc[(df_clean['startYear'] < 1874) | (df_clean['startYear'] > 2030), 'startYear'] = np.nan
        df_clean['decade'] = ((df_clean['startYear'] // 10) * 10)

    # Feature 3: Number of lead actors
    if 'lead_actors_ids' in df_clean.columns:
        def count_lead_actors(value):
            if isinstance(value, list): 
                return len(value)
            text = str(value).strip()
            if text in ['', '[]', '\\N', 'nan', 'None']: 
                return 0
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, list): 
                    return len(parsed)
            except: 
                pass
            if ',' in text: 
                return len([i for i in text.split(',') if i.strip() != ''])
            if ' ' in text: 
                return len([i for i in text.split() if i.strip() != ''])
            return 0
            
        df_clean['num_lead_actors'] = df_clean['lead_actors_ids'].apply(count_lead_actors)

    # Feature 4: Number of genres
    if 'genres' in df_clean.columns:
        def count_genres(value):
            if isinstance(value, list):
                return len(value)
            text = str(value).strip()
            if text in ['', '[]', '\\N', 'Unknown']:
                return 0
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, list):
                    return len(parsed)
            except:
                pass
            if ',' in text:
                split_values = [item.strip() for item in text.split(',') if item.strip() != '']
                return len(split_values)
            if ' ' in text:
                split_values = [item.strip() for item in text.split() if item.strip() != '']
                return len(split_values)
            return 1

        df_clean['num_genres'] = df_clean['genres'].apply(count_genres)

    # --------------------------------------------------------------------------
    # Feature 5: חילוץ בינארי דינמי למדינות, שפות וז'אנרים (Multi-label)
    # --------------------------------------------------------------------------
    
    # --- 10 המדינות המובילות (דינמי) ---
    if 'Country' in df_clean.columns:
        df_clean['Country_parsed'] = df_clean['Country'].apply(parse_categories)
        top_countries = df_clean['Country_parsed'].explode().value_counts().head(10).index.tolist()
        
        created_country_cols = []
        for country in top_countries:
            col_name = f"In_{country.replace('-', '_').replace(' ', '_').title()}"
            created_country_cols.append(col_name)
            df_clean[col_name] = df_clean['Country_parsed'].apply(lambda x: 1 if country in x else 0)
            
        if created_country_cols:
            df_clean['In_Other_Country'] = (df_clean[created_country_cols].sum(axis=1) == 0).astype(int)
            
        df_clean = df_clean.drop(columns=['Country_parsed'])

    # --- 10 השפות המובילות (דינמי) ---
    if 'Language' in df_clean.columns:
        df_clean['Language_parsed'] = df_clean['Language'].apply(parse_categories)
        top_languages = df_clean['Language_parsed'].explode().value_counts().head(10).index.tolist()
        
        for lang in top_languages:
            col_name = f"is_lang_{lang.replace('-', '_').replace(' ', '_')}"
            df_clean[col_name] = df_clean['Language_parsed'].apply(lambda x: 1 if lang in x else 0)
            
        # ללא עמודת is_lang_other למניעת מולטיקולינאריות
        df_clean = df_clean.drop(columns=['Language_parsed'])

    # --- 12 הז'אנרים המובילים (דינמי) ---
    if 'genres' in df_clean.columns:
        df_clean['genres_parsed'] = df_clean['genres'].apply(parse_categories)
        top_genres = df_clean['genres_parsed'].explode().value_counts().head(12).index.tolist()
        
        created_genre_cols = []
        for genre in top_genres:
            col_name = f"is_genre_{genre.replace('-', '_').replace(' ', '_')}"
            created_genre_cols.append(col_name)
            df_clean[col_name] = df_clean['genres_parsed'].apply(lambda x: 1 if genre in x else 0)
            
        if created_genre_cols:
            df_clean['is_genre_other'] = (df_clean[created_genre_cols].sum(axis=1) == 0).astype(int)
            
        df_clean = df_clean.drop(columns=['genres_parsed'])

    # --------------------------------------------------------------------------
    # Feature 6: runtimeMinutes & runtime_category
    # --------------------------------------------------------------------------
    if 'runtimeMinutes' in df_clean.columns:
        df_clean['runtimeMinutes'] = pd.to_numeric(df_clean['runtimeMinutes'], errors='coerce')
        df_clean.loc[(df_clean['runtimeMinutes'] <= 0) | (df_clean['runtimeMinutes'] > 420), 'runtimeMinutes'] = np.nan
        
        def runtime_category(runtime):
            if pd.isna(runtime):
                return 'Unknown'
            if runtime < 60:
                return 'Short'
            elif runtime <= 120:
                return 'Standard'
            else:
                return 'Long'

        df_clean['runtime_category'] = df_clean['runtimeMinutes'].apply(runtime_category)

    # ==========================================================================
    # 3. ניקוי סופי והסרת עמודות מקור גולמיות
    # ==========================================================================
    columns_to_remove = [
        'tconst',
        'primaryTitle',
        'startYear',
        'genres',
        'num_genres',
        'lead_actors_ids',
        'Language',   
        'Country',
        'plot',
        'budget'
    ]
    df_clean = df_clean.drop(columns=[col for col in columns_to_remove if col in df_clean.columns])
    
    return df_clean

In [63]:
df_prepared = prepare_data(df)

In [65]:
print(df_prepared.columns)

Index(['runtimeMinutes', 'averageRating', 'plot_word_count', 'decade',
       'num_lead_actors', 'In_United_States', 'In_India', 'In_United_Kingdom',
       'In_France', 'In_Italy', 'In_Japan', 'In_Canada', 'In_Germany',
       'In_Spain', 'In_Hong_Kong', 'In_Other_Country', 'is_lang_english',
       'is_lang_french', 'is_lang_hindi', 'is_lang_spanish', 'is_lang_italian',
       'is_lang_japanese', 'is_lang_german', 'is_lang_tamil', 'is_lang_telugu',
       'is_lang_malayalam', 'is_genre_drama', 'is_genre_comedy',
       'is_genre_romance', 'is_genre_action', 'is_genre_documentary',
       'is_genre_crime', 'is_genre_thriller', 'is_genre_horror',
       'is_genre_adventure', 'is_genre_mystery', 'is_genre_family',
       'is_genre_fantasy', 'is_genre_other', 'runtime_category'],
      dtype='object')


In [15]:
df_prepared.head(20)

,runtimeMinutes,averageRating,plot_word_count,decade,num_lead_actors,In_USA,In_France,In_UK,In_Italy,In_Spain,...,is_genre_romance,is_genre_action,is_genre_thriller,is_genre_horror,is_genre_crime,is_genre_adventure,is_genre_biography,is_genre_sci_fi,is_genre_animation,runtime_category
0,90.0,6.6,0,1990.0,5,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,Standard
1,86.0,6.1,0,2020.0,5,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Standard
4,102.0,6.1,0,2000.0,5,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Standard
7,85.0,7.0,0,2010.0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,Standard
8,170.0,9.2,0,1990.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Long
9,87.0,7.3,0,2010.0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,Standard
10,83.0,6.3,0,1940.0,5,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Standard
11,81.0,6.3,0,2010.0,5,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Standard
12,130.0,5.8,0,2020.0,5,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,Long
13,107.0,6.8,0,1990.0,5,1,0,1,0,0,...,0,0,0,0,0,0,1,0,0,Standard


<div style="background-color:#d4edda; 
            padding:12px; 
            border-radius:8px; 
            border:1px solid #c3e6cb;
            color:#155724;
            font-weight:bold;
            text-align:center;">

✅ The dataset has been successfully cleaned, transformed, and prepared for machine learning modeling.

</div>

## Feature Analysis: Missing Values, Variance, and Correlation Checks

#### Checking for missing values in the new DF

missing_values = (
    df
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_values = missing_values[missing_values > 0]

missing_values

In [67]:
# Variances for each of our Features and Modeling Data
variance = (
    df_prepared
    .var(numeric_only=True)
    .sort_values()
)

variance

In_Hong_Kong                0.006568
In_Spain                    0.007966
is_lang_malayalam           0.008366
In_Germany                  0.008502
is_lang_telugu              0.008749
is_lang_tamil               0.010175
is_lang_german              0.010895
In_Canada                   0.011892
is_lang_japanese            0.013208
In_Japan                    0.013368
is_lang_italian             0.016280
In_Italy                    0.016782
is_lang_spanish             0.018176
In_France                   0.020039
is_lang_hindi               0.020163
is_lang_french              0.021298
In_United_Kingdom           0.026810
is_genre_other              0.028662
is_genre_fantasy            0.034719
is_genre_family             0.040953
is_genre_mystery            0.045901
In_India                    0.055748
is_genre_adventure          0.066214
is_genre_horror             0.079026
is_genre_thriller           0.091307
is_genre_crime              0.095831
is_genre_documentary        0.099935
i

#### HeatMap for correletion between Features

## Models

#### Compiling and Training our model
#### Check our indicators

In [69]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector # 👈 הייבוא החדש לזיהוי דינמי
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet

# שלב 1: פיצול הדאטה
X = df_prepared.drop(columns=['averageRating'])
y = df_prepared['averageRating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# שלב 2: בניית תת-צינור נומרי (מטפל בכל העמודות המספריות, כולל עמודות ה-0/1)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler())
])

# שלב 3: בניית תת-צינור קטגורי (מטפל בטקסט)
# הוספתי כאן גם Imputer ליתר ביטחון, למקרה שיש ערכי NaN בעמודות הטקסט
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# שלב 4: בניית ה-ColumnTransformer עם בחירה דינמית!
preprocessor = ColumnTransformer(
    transformers=[
        # שואב אוטומטית כל עמודה מספרית (int או float)
        ('num', numeric_transformer, make_column_selector(dtype_include=np.number)),
        
        # שואב אוטומטית כל עמודה שאינה מספרית (טקסט/אובייקט/קטגוריה)
        ('cat', categorical_transformer, make_column_selector(dtype_exclude=np.number))
    ],
    remainder='drop' # מאחר והסלקטורים מכסים 100% מהעמודות, אין שאריות
)

# שלב 5: הגדרת ה-Pipeline עבור Elastic Net
en_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', ElasticNet(random_state=42, max_iter=10000)) # 👈 הוספתי max_iter כדי למנוע אזהרות במודל
])

param_grid_en = {
    'regressor__alpha': [0.01, 0.1, 1.0, 10.0],
    'regressor__l1_ratio': [0.2, 0.5, 0.8]
}

# הרצה במקביל עם 10-Fold CV
grid_search_en = GridSearchCV(
    estimator=en_pipeline, 
    param_grid=param_grid_en, 
    cv=10,
    scoring=['neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2'],
    refit='neg_root_mean_squared_error', 
    n_jobs=-1
)

# שלב 6: אימון המודל
print("מתחיל לאמן את המודל, אנא המתן...")
grid_search_en.fit(X_train, y_train)
print("Elastic Net Fitted Successfully!")

מתחיל לאמן את המודל, אנא המתן...
Elastic Net Fitted Successfully!


In [70]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from IPython.display import display

def evaluate_model_performance(grid_search, X_train, X_test, y_train, y_test, model_name="המודל"):
    """
    פונקציה המפיקה דו"ח ביצועים מלא, כולל תוצאות CV, בדיקת Overfitting וזליגת מידע.
    """
    # ==========================================================================
    # חלק 1: חילוץ ופירוט המדדים עבור Cross Validation באופן דינמי
    # ==========================================================================
    best_idx = grid_search.best_index_
    res = grid_search.cv_results_
    n_splits = grid_search.n_splits_  # 👈 שואב אוטומטית את כמות הפולדס (למשל 10)

    folds_data = {
        'פולד (Fold)': [f"Fold {i+1}" for i in range(n_splits)],
        'RMSE': [],
        'MAE': [],
        'R²': []
    }

    metrics_map = {
        'RMSE': 'neg_root_mean_squared_error',
        'MAE': 'neg_mean_absolute_error',
        'R²': 'r2'
    }

    # לולאה לחילוץ הציונים מכל פולד בנפרד (עד כמות ה-n_splits)
    for fold in range(n_splits):
        rmse_val = abs(res[f'split{fold}_test_neg_root_mean_squared_error'][best_idx])
        mae_val = abs(res[f'split{fold}_test_neg_mean_absolute_error'][best_idx])
        r2_val = res[f'split{fold}_test_r2'][best_idx]
        
        folds_data['RMSE'].append(rmse_val)
        folds_data['MAE'].append(mae_val)
        folds_data['R²'].append(r2_val)

    df_folds = pd.DataFrame(folds_data).set_index('פולד (Fold)')
    print("="*60)
    print(f"--- פירוט מדדי ההערכה לכל אחד מ-{n_splits} הפולדס עבור {model_name} ---")
    print("="*60)
    display(df_folds.round(4))

    print("\n" + "-"*60 + "\n")
    print(f"הפרמטרים הטובים ביותר שנבחרו: {grid_search.best_params_}\n")
    print("--- סיכום סטטיסטי של ה-Cross Validation (ממוצע ± סטיית תקן) ---")

    for name, key in metrics_map.items():
        mean_score = abs(res[f'mean_test_{key}'][best_idx]) if name in ['RMSE', 'MAE'] else res[f'mean_test_{key}'][best_idx]
        std_score = res[f'std_test_{key}'][best_idx]
        print(f"{name}: {mean_score:.4f} (± {std_score:.4f})")

    # ==========================================================================
    # חלק 2: בדיקת זליגת מידע (Data Leakage) והתאמת-יתר (Overfitting)
    # ==========================================================================
    print("\n" + "="*60)
    print(f"--- דו\"ח השוואת אימון (Train) מול מבחן (Test) עבור {model_name} ---")
    print("="*60)

    # 1. הפקת תחזיות באמצעות המודל המנצח
    best_model = grid_search.best_estimator_
    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)

    # 2. חישוב המדדים בפועל (תמיכה מלאה באמצעות np.sqrt)
    train_metrics = {
        'RMSE': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'MAE': mean_absolute_error(y_train, y_train_pred),
        'R²': r2_score(y_train, y_train_pred)
    }

    test_metrics = {
        'RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'MAE': mean_absolute_error(y_test, y_test_pred),
        'R²': r2_score(y_test, y_test_pred)
    }

    # 3. יצירת טבלת השוואה מסכמת
    df_leakage_check = pd.DataFrame({
        'אימון (Train Set)': [train_metrics['RMSE'], train_metrics['MAE'], train_metrics['R²']],
        'מבחן (Test Set)': [test_metrics['RMSE'], test_metrics['MAE'], test_metrics['R²']]
    }, index=['RMSE (טעות)', 'MAE (טעות)', 'R² (אחוז הסבר)'])

    display(df_leakage_check.round(4))
    print("\n" + "-"*60 + "\n")

    # ==========================================================================
    # חלק 3: ניתוח אוטומטי ומסקנות סטטיסטיות מבוססות חוקים
    # ==========================================================================
    print("🔍 ניתוח ומסקנות סטטיסטיות מהדו\"ח:")

    r2_train = train_metrics['R²']
    r2_test = test_metrics['R²']
    r2_diff = r2_train - r2_test

    # א. בדיקת זליגת מידע (Data Leakage)
    if r2_train > 0.95 or r2_test > 0.95:
        print("🚨 אזהרה: מדד ה-R² גבוה באופן קיצוני (מעל 95%). יש חשד חמור ל-Data Leakage!")
        print("   ודא שלא נשארו בדאטה עמודות זליגה שהיו אמורות להימחק, כגון מספר מצביעים או הכנסות.")
    else:
        print("✅ בדיקת זליגה: תקין! מדד ה-R² הגיוני ואינו מושלם מדי. אין סימנים לזליגת מידע (Data Leakage).")

    # ב. בדיקת התאמת-יתר (Overfitting)
    if r2_diff > 0.15:
        print(f"🚨 אזהרה: יש פער משמעותי של {r2_diff:.2%} ב-R² לטובת ה-Train. המודל סובל מ-Overfitting חריף!")
        print("   המודל למד את הרעש של נתוני האימון ומתקשה להכליל על נתונים חדשים.")
    elif r2_diff < 0:
        print(f"✅ בדיקת יציבות: מדד ה-Test מעט טוב יותר או שווה ל-Train (פער של {r2_diff:.2%}).")
        print("   זוהי תופעה מצוינת המעידה על מודל חסין, יציב ומכליל היטב (Generalization).")
    else:
        print(f"✅ בדיקת התאמת-יתר: תקין! הפער בין ה-Train ל-Test קטן מאוד ({r2_diff:.2%}).")
        print("   המודל שומר על יציבות ואינו סובל מ-Overfitting.")
    print("="*60)

In [71]:
# שורת ההפעלה שמייצרת את הדו"ח בפועל:
evaluate_model_performance(grid_search_en, X_train, X_test, y_train, y_test, model_name="Elastic Net")

--- פירוט מדדי ההערכה לכל אחד מ-10 הפולדס עבור Elastic Net ---


,RMSE,MAE,R²
פולד (Fold),,,
Fold 1,1.1280,0.8665,0.2280
Fold 2,1.1434,0.8764,0.2355
Fold 3,1.1336,0.8692,0.2319
Fold 4,1.1230,0.8667,0.2484
Fold 5,1.1467,0.8697,0.2227
Fold 6,1.1444,0.8744,0.2297
Fold 7,1.1198,0.8600,0.2390
Fold 8,1.1308,0.8672,0.2411
Fold 9,1.1370,0.8728,0.2380



------------------------------------------------------------

הפרמטרים הטובים ביותר שנבחרו: {'regressor__alpha': 0.01, 'regressor__l1_ratio': 0.2}

--- סיכום סטטיסטי של ה-Cross Validation (ממוצע ± סטיית תקן) ---
RMSE: 1.1309 (± 0.0128)
MAE: 0.8669 (± 0.0083)
R²: 0.2364 (± 0.0081)

--- דו"ח השוואת אימון (Train) מול מבחן (Test) עבור Elastic Net ---


,אימון (Train Set),מבחן (Test Set)
RMSE (טעות),1.1306,1.1285
MAE (טעות),0.8665,0.8637
R² (אחוז הסבר),0.2370,0.2308



------------------------------------------------------------

🔍 ניתוח ומסקנות סטטיסטיות מהדו"ח:
✅ בדיקת זליגה: תקין! מדד ה-R² הגיוני ואינו מושלם מדי. אין סימנים לזליגת מידע (Data Leakage).
✅ בדיקת התאמת-יתר: תקין! הפער בין ה-Train ל-Test קטן מאוד (0.62%).
   המודל שומר על יציבות ואינו סובל מ-Overfitting.


In [86]:
X = df_prepared.drop(columns=['averageRating'])
y = df_prepared['averageRating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================================================
# 2. הגדרת צינורות העיבוד (Pipelines) אוטומטיים
# ==========================================================================
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# ה-ColumnTransformer שואב אוטומטית את העמודות לפי הסוג שלהן
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, make_column_selector(dtype_include=np.number)),
        ('cat', categorical_transformer, make_column_selector(dtype_exclude=np.number))
    ],
    remainder='drop'
)

# ==========================================================================
# 3. צינור האימון של Random Forest
# ==========================================================================
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1)) # n_jobs=-1 מאיץ משמעותית את החישוב!
])

# פרמטרים בסיסיים ויעילים לבדיקה כדי שלא ייקח נצח
param_grid_rf = {
    'regressor__n_estimators': [100, 200],  # כמות העצים ביער
    'regressor__max_depth': [10, 20, None]  # עומק העץ
}

grid_search_rf = GridSearchCV(
    estimator=rf_pipeline, 
    param_grid=param_grid_rf, 
    cv=10, 
    scoring=['neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2'],
    refit='neg_root_mean_squared_error', 
    n_jobs=-1
)

# ==========================================================================
# 4. הרצה והדפסת התוצאות
# ==========================================================================
print("מתחיל לאמן את מודל Random Forest על כל הדאטה... (זה עשוי לקחת כמה דקות ☕)")
grid_search_rf.fit(X_train, y_train)
print("אימון ה-Random Forest הושלם בהצלחה!\n")

# קורא לפונקציה המוכרת שלנו כדי להציג את הדו"ח (הנחתי שהיא כבר הורצה בזיכרון קודם)
evaluate_model_performance(grid_search_rf, X_train, X_test, y_train, y_test, model_name="Random Forest")

מתחיל לאמן את מודל Random Forest על כל הדאטה... (זה עשוי לקחת כמה דקות ☕)
אימון ה-Random Forest הושלם בהצלחה!

--- פירוט מדדי ההערכה לכל אחד מ-10 הפולדס עבור Random Forest ---


,RMSE,MAE,R²
פולד (Fold),,,
Fold 1,1.1060,0.8441,0.2579
Fold 2,1.1234,0.8543,0.2620
Fold 3,1.1114,0.8464,0.2617
Fold 4,1.1037,0.8491,0.2740
Fold 5,1.1250,0.8505,0.2518
Fold 6,1.1231,0.8552,0.2581
Fold 7,1.0972,0.8388,0.2694
Fold 8,1.1094,0.8437,0.2696
Fold 9,1.1139,0.8493,0.2687



------------------------------------------------------------

הפרמטרים הטובים ביותר שנבחרו: {'regressor__max_depth': 10, 'regressor__n_estimators': 200}

--- סיכום סטטיסטי של ה-Cross Validation (ממוצע ± סטיית תקן) ---
RMSE: 1.1092 (± 0.0132)
MAE: 0.8458 (± 0.0080)
R²: 0.2654 (± 0.0081)

--- דו"ח השוואת אימון (Train) מול מבחן (Test) עבור Random Forest ---


,אימון (Train Set),מבחן (Test Set)
RMSE (טעות),1.0866,1.1087
MAE (טעות),0.8281,0.8445
R² (אחוז הסבר),0.2952,0.2576



------------------------------------------------------------

🔍 ניתוח ומסקנות סטטיסטיות מהדו"ח:
✅ בדיקת זליגה: תקין! מדד ה-R² הגיוני ואינו מושלם מדי. אין סימנים לזליגת מידע (Data Leakage).
✅ בדיקת התאמת-יתר: תקין! הפער בין ה-Train ל-Test קטן מאוד (3.76%).
   המודל שומר על יציבות ואינו סובל מ-Overfitting.
